# Practical 1 - Matrix Operations with Pandas and NumPy
Dataset: Teaching Ratings (463 course evaluations of 94 instructors)

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
def find_data(name):
    """Look for the CSV in the usual places; in Colab, ask for an upload if it is missing."""
    folders = (HERE.parent / "Datasets", HERE / "Datasets", HERE, Path.cwd(),
               Path("/content"), Path("/content/Datasets"), Path("/content/drive/MyDrive"))
    for folder in folders:
        if (folder / name).exists():
            return folder
    try:
        from google.colab import files
        print(f"{name} not found - please choose it in the upload dialog (it is in the Datasets folder).")
        files.upload()
        return Path.cwd()
    except ImportError:
        raise FileNotFoundError(f"Could not find {name}. Put it in a 'Datasets' folder next to this script.")

DATA = find_data("TeachingRatings.csv")

ratings = pd.read_csv(DATA / "TeachingRatings.csv")
np.set_printoptions(precision=4, suppress=True)

print("Rows x Columns :", ratings.shape)
print("\nColumn types:")
print(ratings.dtypes.to_string())
print("\nMissing values per column:", int(ratings.isna().sum().sum()))

Rows x Columns : (463, 13)

Column types:
rownames         int64
minority           str
age              int64
gender             str
credits            str
beauty         float64
eval           float64
division           str
native             str
tenure             str
students         int64
allstudents      int64
prof             int64

Missing values per column: 0


## 1. From DataFrame to NumPy array

In [2]:
numeric_cols = ["age", "beauty", "eval", "students"]
data = ratings[numeric_cols].to_numpy(dtype=float)

print("Array shape      :", data.shape)
print("Number of items  :", data.size)
print("Dimensions (ndim):", data.ndim)
print("Data type        :", data.dtype)
print("Memory used      :", data.nbytes, "bytes")

Array shape      : (463, 4)
Number of items  : 1852
Dimensions (ndim): 2
Data type        : float64
Memory used      : 14816 bytes


## 2. Pick two small blocks of rows and form square matrices

In [3]:
subset = ratings.iloc[4:8][["age", "students"]]
print("Selected rows (4 to 7), columns age & students:")
print(subset.to_string())

block = subset.to_numpy(dtype=float)
P, Q = block[:2], block[2:]
print("\nMatrix P (rows 4-5):\n", P)
print("\nMatrix Q (rows 6-7):\n", Q)

Selected rows (4 to 7), columns age & students:
   age  students
4   31        42
5   62       182
6   33        33
7   51        25

Matrix P (rows 4-5):
 [[ 31.  42.]
 [ 62. 182.]]

Matrix Q (rows 6-7):
 [[33. 33.]
 [51. 25.]]


## 3. Element-wise operations

In [4]:
operations = {
    "P + Q": P + Q,
    "P - Q": P - Q,
    "P * Q (element-wise)": P * Q,
    "P / Q (element-wise)": P / Q,
}
for label, result in operations.items():
    print(f"{label}:\n{result}\n")

P + Q:
[[ 64.  75.]
 [113. 207.]]

P - Q:
[[ -2.   9.]
 [ 11. 157.]]

P * Q (element-wise):
[[1023. 1386.]
 [3162. 4550.]]

P / Q (element-wise):
[[0.9394 1.2727]
 [1.2157 7.28  ]]



## 4. Linear-algebra operations

In [5]:
print("Matrix product P @ Q:\n", P @ Q)
print("\nMatrix product Q @ P:\n", Q @ P)
print("\nTranspose of P:\n", P.T)
print("\nDeterminant of P :", round(np.linalg.det(P), 4))
print("Rank of P        :", np.linalg.matrix_rank(P))

Matrix product P @ Q:
 [[ 3165.  2073.]
 [11328.  6596.]]

Matrix product Q @ P:
 [[3069. 7392.]
 [3131. 6692.]]

Transpose of P:
 [[ 31.  62.]
 [ 42. 182.]]

Determinant of P : 3038.0
Rank of P        : 2


## 5. Inverse and verification

In [6]:
def safe_inverse(M):
    if abs(np.linalg.det(M)) < 1e-12:
        raise ValueError("Matrix is singular, no inverse exists")
    return np.linalg.inv(M)

P_inv = safe_inverse(P)
print("Inverse of P:\n", P_inv)
print("\nCheck P @ P_inv (should be identity):\n", np.round(P @ P_inv, 6))
print("Identity recovered:", np.allclose(P @ P_inv, np.eye(2)))

Inverse of P:
 [[ 0.0599 -0.0138]
 [-0.0204  0.0102]]

Check P @ P_inv (should be identity):
 [[ 1.  0.]
 [-0.  1.]]
Identity recovered: True


## 6. Column statistics straight from the array

In [7]:
summary = pd.DataFrame({
    "mean": data.mean(axis=0),
    "std": data.std(axis=0, ddof=1),
    "min": data.min(axis=0),
    "max": data.max(axis=0),
}, index=numeric_cols)
print(summary.round(3).to_string())

            mean     std    min     max
age       48.365   9.803  29.00   73.00
beauty     0.000   0.789  -1.45    1.97
eval       3.998   0.555   2.10    5.00
students  36.624  45.018   5.00  380.00


## 7. Encoding the text columns (one-hot)

In [8]:
text_cols = ratings.select_dtypes(exclude="number").columns.tolist()
print("Categorical columns:", text_cols)

encoded = pd.get_dummies(ratings, columns=text_cols, dtype=int)
print("\nShape before encoding:", ratings.shape)
print("Shape after encoding :", encoded.shape)
print("\nFirst 3 rows of the encoded columns:")
dummy_cols = [c for c in encoded.columns if c not in ratings.columns]
print(encoded[dummy_cols].head(3).to_string())

Categorical columns: ['minority', 'gender', 'credits', 'division', 'native', 'tenure']

Shape before encoding: (463, 13)
Shape after encoding : (463, 19)

First 3 rows of the encoded columns:
   minority_no  minority_yes  gender_female  gender_male  credits_more  credits_single  division_lower  division_upper  native_no  native_yes  tenure_no  tenure_yes
0            0             1              1            0             1               0               0               1          0           1          0           1
1            1             0              0            1             1               0               0               1          0           1          0           1
2            1             0              0            1             1               0               0               1          0           1          0           1
